# 🏆 Insurance Claims Forecasting
### Predict Monthly Claim Frequency, Severity & Total (Aug–Dec 2025)

---

| Item | Detail |
|------|--------|
| **Target** | Claim Frequency, Claim Severity, Total Claim |
| **Horizon** | August – December 2025 (5 months) |
| **Granularity** | Weekly → Monthly rollup |
| **Evaluation** | MAPE (Mean Absolute Percentage Error) |

**Notebook Sections**
1. Setup & Data Loading  
2. Data Preparation  
3. Feature Engineering & Transformation  
4. Time-Based Cross Validation  
5. Baseline Time-Series Models  
6. Classic Statistical Models (SES, ARIMA, SARIMA, Theta)  
7. Gradient Boosting Model (LightGBM)  
8. Linear Decomposition Models (NLinear, DLinear)  
9. Ensemble Learning & Model Blending  
10. Final Predictions & Submission

---
## 1. Setup & Data Loading

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import warnings
import os
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Data & Math ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from numpy.linalg import lstsq
from scipy.optimize import minimize_scalar, minimize
from scipy.stats import mstats

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("[INFO] LightGBM not found — Gradient Boosting section will be skipped.")

# ── Statistical ───────────────────────────────────────────────────────────────
try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from pmdarima import auto_arima
    HAS_STATSMODELS = True
    HAS_PMDARIMA = True
except ImportError:
    HAS_STATSMODELS = False
    HAS_PMDARIMA = False
    print("[INFO] statsmodels/pmdarima not found — ARIMA/SARIMA will use fallback.")

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.35,
    'font.size': 11,
})

np.random.seed(42)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR   = Path('data')
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Setup complete.')
print(f'  Data dir   : {DATA_DIR.resolve()}')
print(f'  Output dir : {OUTPUT_DIR.resolve()}')

In [ ]:
# ── Raw Load ──────────────────────────────────────────────────────────────────
df_klaim = pd.read_csv(DATA_DIR / 'Data_Klaim.csv')
df_polis = pd.read_csv(DATA_DIR / 'Data_Polis.csv')

print(f'Data_Klaim : {df_klaim.shape[0]:,} rows × {df_klaim.shape[1]} cols')
print(f'Data_Polis : {df_polis.shape[0]:,} rows × {df_polis.shape[1]} cols')
print()
df_klaim.head(3)

---
## 2. Data Preparation

### 2.1 Parsing & Typing

In [ ]:
# ── Parse dates & amounts ─────────────────────────────────────────────────────
df_klaim['date']   = pd.to_datetime(df_klaim['Tanggal Pasien Masuk RS'])
df_klaim['amount'] = pd.to_numeric(df_klaim['Nominal Klaim Yang Disetujui'], errors='coerce')
df_klaim['ym']     = list(zip(df_klaim['date'].dt.year, df_klaim['date'].dt.month))
df_klaim['week']   = df_klaim['date'].dt.to_period('W')

print('Date range :', df_klaim['date'].min().date(), '→', df_klaim['date'].max().date())
print('Null amounts:', df_klaim['amount'].isna().sum())
df_klaim[['date','amount']].describe()

### 2.2 Monthly Aggregation (Ground Truth)

In [ ]:
# ── Monthly ground truth ──────────────────────────────────────────────────────
monthly = (
    df_klaim
    .groupby('ym')
    .agg(frequency=('Claim ID', 'count'),
         total_claim=('amount', 'sum'))
    .reset_index()
)
monthly['severity'] = monthly['total_claim'] / monthly['frequency']
monthly = monthly.sort_values('ym').reset_index(drop=True)
monthly['date'] = monthly['ym'].apply(lambda x: pd.Timestamp(year=x[0], month=x[1], day=1))

print(f'Monthly rows : {len(monthly)}')
print(f'Period       : {monthly["ym"].iloc[0]} → {monthly["ym"].iloc[-1]}')
monthly[['date','frequency','severity','total_claim']].tail(8)

### 2.3 Weekly Aggregation

In [ ]:
# ── Weekly aggregation ────────────────────────────────────────────────────────
weekly = (
    df_klaim
    .groupby('week')
    .agg(frequency=('Claim ID', 'count'),
         total_claim=('amount', 'sum'))
    .reset_index()
)
weekly['week_start'] = weekly['week'].dt.start_time.dt.normalize()
weekly['week_end']   = weekly['week'].dt.end_time.dt.normalize()
weekly = weekly.sort_values('week_start').reset_index(drop=True)

print(f'Weekly rows : {len(weekly)}')
print(f'Period      : {weekly["week_start"].iloc[0].date()} → {weekly["week_end"].iloc[-1].date()}')
weekly.tail(5)

### 2.4 Rollup Utility  
_Convert weekly predictions → monthly totals using day-fraction weighting._

In [ ]:
def weekly_to_monthly_rollup(ws_list, we_list, values, target_yms):
    """
    Sum weekly predictions into monthly totals.
    Each day inside a week contributes  value / 7  to its month.
    Partial weeks at month boundaries are handled automatically.
    """
    monthly_sums = {ym: 0.0 for ym in target_yms}
    for ws, we, val in zip(ws_list, we_list, values):
        for day in pd.date_range(ws, we, freq='D'):
            ym = (day.year, day.month)
            if ym in monthly_sums:
                monthly_sums[ym] += val / 7.0
    return monthly_sums


def get_future_weeks(month_start, month_end):
    """Generate (week_start, week_end) pairs covering a calendar month."""
    fws, fwe = [], []
    cursor = month_start
    while cursor <= month_end:
        fws.append(cursor)
        fwe.append(min(cursor + pd.Timedelta(days=6), month_end))
        cursor += pd.Timedelta(days=7)
    return fws, fwe


print('Rollup utilities defined.')

---
## 3. Feature Engineering & Transformation

### 2.4.1 Lag Features

In [ ]:
def build_lag_features(series: np.ndarray, lookback: int, target_col: str = 'y'):
    """
    Build a supervised learning DataFrame with lag features.

    Parameters
    ----------
    series   : 1-D array of time-series values
    lookback : number of lag steps to include
    target_col : name for the target column

    Returns
    -------
    X : np.ndarray  shape (n - lookback, lookback)
    y : np.ndarray  shape (n - lookback,)
    """
    series = np.array(series, dtype=float)
    n = len(series)
    X = np.stack([series[i : i + lookback] for i in range(n - lookback)])
    y = series[lookback:]
    return X, y


def build_lag_dataframe(series: np.ndarray, lookback: int, target_col: str = 'y'):
    """Same as build_lag_features but returns a tidy DataFrame."""
    X, y = build_lag_features(series, lookback, target_col)
    cols = [f'lag_{i+1}' for i in range(lookback)]
    df = pd.DataFrame(X, columns=cols)
    df[target_col] = y
    return df


# ── Quick demo ────────────────────────────────────────────────────────────────
demo = build_lag_dataframe(weekly['frequency'].values, lookback=4)
print(f'Lag feature matrix shape: {demo.shape}')
demo.head(4)

### 2.4.2 Logarithmic Transformation (Log-Scaling)

In [ ]:
def winsorize_series(arr: np.ndarray, n_std: float = 2.5) -> np.ndarray:
    """Cap values at mean ± n_std × std (in-place safe)."""
    arr = np.array(arr, dtype=float)
    mu, sigma = arr.mean(), arr.std()
    return np.clip(arr, mu - n_std * sigma, mu + n_std * sigma)


def log_transform(arr: np.ndarray, floor: float = 1.0) -> np.ndarray:
    """Natural log with floor to avoid log(0)."""
    return np.log(np.maximum(np.array(arr, dtype=float), floor))


def exp_transform(arr: np.ndarray) -> np.ndarray:
    """Inverse of log_transform."""
    return np.exp(np.array(arr, dtype=float))


def prepare_series(arr: np.ndarray,
                   use_log: bool = False,
                   winsorize: bool = True,
                   n_std: float = 2.5) -> np.ndarray:
    """Full preprocessing pipeline: winsorize → optional log."""
    arr = np.array(arr, dtype=float)
    if winsorize:
        arr = winsorize_series(arr, n_std)
    if use_log:
        arr = log_transform(arr)
    return arr


# ── Visualise the effect ──────────────────────────────────────────────────────
tot_raw = weekly['total_claim'].values.astype(float)
tot_log = log_transform(winsorize_series(tot_raw))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].plot(tot_raw / 1e9, color='steelblue')
axes[0].set_title('Weekly Total Claim  (raw, B IDR)')
axes[1].plot(tot_log, color='darkorange')
axes[1].set_title('Weekly Total Claim  (log-transformed, winsorised)')
for ax in axes:
    ax.set_xlabel('Week index')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_log_transform.png', bbox_inches='tight')
plt.show()

print(f'Raw  — skew: {pd.Series(tot_raw).skew():.3f} | CV: {tot_raw.std()/tot_raw.mean():.3f}')
print(f'Log  — skew: {pd.Series(tot_log).skew():.3f} | CV: {tot_log.std()/tot_log.mean():.3f}')

---
## 4. Time-Based Cross Validation (Walk-Forward)

### 2.12.1 Walk-Forward Validation

Train on all data **before** each test month → predict that month → evaluate.

In [ ]:
# ── Shared CV infrastructure ──────────────────────────────────────────────────
CV_N_TEST_MONTHS = 5   # last 5 months = Mar–Jul 2025

MONTHLY_GT   = {r['ym']: r for _, r in monthly.iterrows()}
ALL_MONTHS   = sorted(monthly['ym'].tolist())
HELD_OUT     = ALL_MONTHS[-CV_N_TEST_MONTHS:]
WEEKLY_WE_PD = pd.DatetimeIndex(weekly['week_end'])

FREQ_RAW = weekly['frequency'].values.astype(float)
TOT_RAW  = weekly['total_claim'].values.astype(float)

print('Walk-forward CV setup')
print(f'  Test months ({CV_N_TEST_MONTHS}): {HELD_OUT}')

### 2.12.2 Metrics

In [ ]:
def mape(actual: np.ndarray, predicted: np.ndarray) -> float:
    a, p = np.array(actual, float), np.array(predicted, float)
    mask = a != 0
    return float(np.mean(np.abs((a[mask] - p[mask]) / a[mask])) * 100)


def evaluate_monthly_rollup(pred_freq, pred_total, gt_row) -> dict:
    """Return MAPE dict for frequency, severity, and total."""
    pred_sev = pred_total / pred_freq if pred_freq > 0 else 0.0
    return {
        'freq_ape': abs(gt_row['frequency']   - pred_freq)  / gt_row['frequency']   * 100,
        'sev_ape':  abs(gt_row['severity']    - pred_sev)   / gt_row['severity']    * 100,
        'tot_ape':  abs(gt_row['total_claim'] - pred_total) / gt_row['total_claim'] * 100,
    }


def print_cv_summary(name: str, results: list):
    """Pretty-print walk-forward CV results."""
    hdr = f"{'Month':<14}{'APE_Freq':>10}{'APE_Sev':>10}{'APE_Tot':>10}{'Overall':>10}"
    print(f'\n── {name} CV Results ──')
    print(hdr)
    print('─' * len(hdr))
    all_apes = []
    for r in results:
        ov = (r['freq_ape'] + r['sev_ape'] + r['tot_ape']) / 3
        all_apes += [r['freq_ape'], r['sev_ape'], r['tot_ape']]
        print(f"{str(r['ym']):<14}"
              f"{r['freq_ape']:>9.2f}%"
              f"{r['sev_ape']:>9.2f}%"
              f"{r['tot_ape']:>9.2f}%"
              f"{ov:>9.2f}%")
    print('─' * len(hdr))
    print(f"{'MEAN MAPE':<14}"
          f"{np.mean([r['freq_ape'] for r in results]):>9.2f}%"
          f"{np.mean([r['sev_ape']  for r in results]):>9.2f}%"
          f"{np.mean([r['tot_ape']  for r in results]):>9.2f}%"
          f"{np.mean(all_apes):>9.2f}%")
    return np.mean(all_apes)


def run_model_cv(model_fn, name: str, use_log_total: bool = True):
    """
    Generic walk-forward CV runner.

    Parameters
    ----------
    model_fn : callable(train_series) → object with .predict_ar(h) method
    name     : display name for the model
    use_log_total : whether total_claim is log-transformed

    Returns
    -------
    results  : list of per-month dicts
    mape_ov  : float, overall mean MAPE
    """
    results = []
    for ym in HELD_OUT:
        ms   = pd.Timestamp(year=ym[0], month=ym[1], day=1)
        me   = ms + pd.offsets.MonthEnd(0)
        mask = WEEKLY_WE_PD < ms
        if mask.sum() < 6:
            continue

        tr_f = prepare_series(FREQ_RAW[mask], use_log=False)
        tr_t = prepare_series(TOT_RAW[mask],  use_log=use_log_total)
        fws, fwe = get_future_weeks(ms, me)
        h = len(fws)

        try:
            pf = np.maximum(model_fn(tr_f).predict_ar(h), 0)
            pt = model_fn(tr_t).predict_ar(h)
            if use_log_total:
                pt = exp_transform(pt)

            rof = weekly_to_monthly_rollup(fws, fwe, pf, {ym})
            rot = weekly_to_monthly_rollup(fws, fwe, pt, {ym})

            gt  = MONTHLY_GT[ym]
            res = evaluate_monthly_rollup(rof[ym], rot[ym], gt)
            res['ym'] = ym
            results.append(res)
        except Exception as exc:
            print(f'  [WARN] {name} failed on {ym}: {exc}')

    mape_ov = print_cv_summary(name, results)
    return results, mape_ov


print('CV utilities defined.')

---
## 5. Baseline Time-Series Models

### 2.7.1 Naïve

In [ ]:
class NaiveModel:
    """Forecast = last observed value (random walk)."""

    def fit(self, y: np.ndarray):
        self._last = float(np.array(y)[-1])
        self._y = np.array(y, float).copy()
        return self

    def predict_ar(self, h: int) -> np.ndarray:
        return np.full(h, self._last)


cv_naive, mape_naive = run_model_cv(lambda y: NaiveModel().fit(y), name='Naive')

### 2.7.2 Seasonal Naïve

In [ ]:
def seasonal_naive_predict(train_mg: pd.DataFrame, target_ym: tuple):
    """
    Same-month-last-year value × median YoY ratio of the last 6 months.

    Returns (pred_freq, pred_total) or (None, None) if prior year is missing.
    """
    prev_ym  = (target_ym[0] - 1, target_ym[1])
    prev_row = train_mg[train_mg['ym'] == prev_ym]
    if len(prev_row) == 0:
        return None, None

    yoys_f, yoys_t = [], []
    for offset in range(1, 7):
        m = target_ym[1] - offset
        cur_ym = (target_ym[0],     m) if m > 0 else (target_ym[0] - 1, m + 12)
        prv_ym = (target_ym[0] - 1, m) if m > 0 else (target_ym[0] - 2, m + 12)
        cur = train_mg[train_mg['ym'] == cur_ym]
        prv = train_mg[train_mg['ym'] == prv_ym]
        if len(cur) > 0 and len(prv) > 0:
            pf, pt = float(prv.iloc[0]['freq']), float(prv.iloc[0]['tot'])
            if pf > 0: yoys_f.append(float(cur.iloc[0]['freq']) / pf)
            if pt > 0: yoys_t.append(float(cur.iloc[0]['tot'])  / pt)

    yoy_f = np.median(yoys_f) if yoys_f else 1.0
    yoy_t = np.median(yoys_t) if yoys_t else 1.0
    return float(prev_row.iloc[0]['freq']) * yoy_f, float(prev_row.iloc[0]['tot']) * yoy_t


# ── CV for Seasonal Naïve ─────────────────────────────────────────────────────
mg_agg = monthly.rename(columns={'frequency': 'freq', 'total_claim': 'tot'})
sn_results = []
for ym in HELD_OUT:
    idx     = list(monthly['ym']).index(ym)
    tr_mg   = mg_agg.iloc[:idx]
    pf, pt  = seasonal_naive_predict(tr_mg, ym)
    if pf is None:
        continue
    gt  = MONTHLY_GT[ym]
    res = evaluate_monthly_rollup(pf, pt, gt)
    res['ym'] = ym
    sn_results.append(res)

mape_sn = print_cv_summary('Seasonal Naïve', sn_results)

---
## 6. Classic Statistical Time-Series Models

### 2.8.1 Simple Exponential Smoothing (SES)

In [ ]:
class SESModel:
    """Simple Exponential Smoothing — alpha optimised via SSE minimisation."""

    def fit(self, y: np.ndarray):
        y = np.array(y, float)

        def sse(alpha):
            s, e = y[0], 0.0
            for v in y[1:]:
                e += (v - s) ** 2
                s  = alpha * v + (1 - alpha) * s
            return e

        res          = minimize_scalar(sse, bounds=(0.01, 0.99), method='bounded')
        self.alpha   = res.x
        self._level  = y[0]
        for v in y[1:]:
            self._level = self.alpha * v + (1 - self.alpha) * self._level
        self._y = y.copy()
        return self

    def predict_ar(self, h: int) -> np.ndarray:
        """Auto-regressive: refit on growing buffer for each step."""
        buf, out = list(self._y), []
        for _ in range(h):
            m = SESModel().fit(np.array(buf))
            out.append(m._level)
            buf.append(m._level)
        return np.array(out)


cv_ses, mape_ses = run_model_cv(lambda y: SESModel().fit(y), name='SES')

### 2.8.2 ARIMA

In [ ]:
class ARIMAModel:
    """
    ARIMA(p,d,q) wrapper.
    Uses statsmodels if available; falls back to a simple AR(1) on differences.
    """

    def __init__(self, order=(1, 1, 0)):
        self.order = order

    def fit(self, y: np.ndarray):
        self._y = np.array(y, float).copy()
        if HAS_STATSMODELS:
            self._model = ARIMA(self._y, order=self.order).fit()
        return self

    def predict_ar(self, h: int) -> np.ndarray:
        if HAS_STATSMODELS:
            fc = self._model.forecast(steps=h)
            return np.array(fc)
        # Fallback: AR(1) on first differences
        buf, out = list(self._y), []
        for _ in range(h):
            yy  = np.array(buf)
            dy  = np.diff(yy)
            if len(dy) >= 2:
                X   = dy[:-1].reshape(-1, 1)
                Xa  = np.c_[X, np.ones(len(X))]
                W, *_ = lstsq(Xa, dy[1:], rcond=None)
                dp  = float(W[0] * dy[-1] + W[1])
            else:
                dp = 0.0
            p = yy[-1] + dp
            out.append(p)
            buf.append(p)
        return np.array(out)


cv_arima, mape_arima = run_model_cv(lambda y: ARIMAModel((1, 1, 0)).fit(y), name='ARIMA(1,1,0)')

### 2.8.3 SARIMA & Auto-SARIMA

In [ ]:
class SARIMAModel:
    """
    SARIMA(p,d,q)(P,D,Q,s) wrapper.
    Auto-selects orders via pmdarima if available.
    """

    def __init__(self, order=(1, 1, 0), seasonal_order=(0, 0, 0, 4), auto: bool = False):
        self.order          = order
        self.seasonal_order = seasonal_order
        self.auto           = auto

    def fit(self, y: np.ndarray):
        self._y = np.array(y, float).copy()

        if self.auto and HAS_PMDARIMA:
            self._model = auto_arima(
                self._y, seasonal=True, m=4,
                stepwise=True, suppress_warnings=True,
                error_action='ignore'
            )
        elif HAS_STATSMODELS:
            self._model = SARIMAX(
                self._y,
                order=self.order,
                seasonal_order=self.seasonal_order
            ).fit(disp=False)
        return self

    def predict_ar(self, h: int) -> np.ndarray:
        if (self.auto and HAS_PMDARIMA) or HAS_STATSMODELS:
            if self.auto and HAS_PMDARIMA:
                return np.array(self._model.predict(n_periods=h))
            return np.array(self._model.forecast(steps=h))
        # Fallback to SES
        return SESModel().fit(self._y).predict_ar(h)


cv_sarima, mape_sarima = run_model_cv(
    lambda y: SARIMAModel(auto=HAS_PMDARIMA).fit(y),
    name='Auto-SARIMA'
)

### 2.8.4 Theta Model

In [ ]:
class ThetaModel:
    """
    Theta decomposition method.
    Theta-0 line : OLS trend  (captures long-run trend)
    Theta-2 line : SES        (captures level)
    Forecast = 0.5 * theta0_forecast + 0.5 * theta2_forecast
    """

    def fit(self, y: np.ndarray):
        y  = np.array(y, float)
        n  = len(y)
        t  = np.arange(n, dtype=float)

        # Theta-0: OLS linear trend
        A                   = np.c_[t, np.ones(n)]
        W, *_               = lstsq(A, y, rcond=None)
        self.slope          = float(W[0])
        self.intercept      = float(W[1])

        # Theta-2: SES with optimised alpha
        best_a, best_sse = 0.2, np.inf
        for a in np.linspace(0.05, 0.95, 19):
            s = y[0]
            sse = sum((v - s) ** 2 for v in y[1:] if (s := a * v + (1 - a) * s) or True)
            # clean computation:
        for a in np.linspace(0.05, 0.95, 19):
            s, e = y[0], 0.0
            for v in y[1:]:
                e += (v - s) ** 2
                s  = a * v + (1 - a) * s
            if e < best_sse:
                best_sse = e
                best_a   = a

        self._ses = y[0]
        for v in y[1:]:
            self._ses = best_a * v + (1 - best_a) * self._ses

        self._n = n
        self._y = y.copy()
        return self

    def _predict_one(self) -> float:
        t0 = self.slope * self._n + self.intercept
        return 0.5 * t0 + 0.5 * self._ses

    def predict_ar(self, h: int) -> np.ndarray:
        buf, out = list(self._y), []
        for _ in range(h):
            m = ThetaModel().fit(np.array(buf))
            p = m._predict_one()
            out.append(p)
            buf.append(p)
        return np.array(out)


cv_theta, mape_theta = run_model_cv(lambda y: ThetaModel().fit(y), name='Theta')

---
## 7. Gradient Boosting Time-Series Model

### 2.9.1 Lag-Featured LightGBM

In [ ]:
class LightGBMLagModel:
    """
    LightGBM with auto-regressive lag features.
    Falls back to DLinear if LightGBM is unavailable.
    """

    def __init__(self, lookback: int = 12, lgb_params: dict = None):
        self.lookback   = lookback
        self.lgb_params = lgb_params or {
            'objective':       'regression',
            'metric':          'mape',
            'learning_rate':    0.05,
            'num_leaves':       16,
            'n_estimators':    200,
            'min_child_samples': 5,
            'verbose':         -1,
        }

    def fit(self, y: np.ndarray):
        self._y = np.array(y, float).copy()
        if not HAS_LGB:
            return self
        X, target = build_lag_features(self._y, self.lookback)
        self._model = lgb.LGBMRegressor(**self.lgb_params)
        self._model.fit(X, target)
        return self

    def predict_ar(self, h: int) -> np.ndarray:
        if not HAS_LGB:
            # Fallback: SES
            return SESModel().fit(self._y).predict_ar(h)
        buf, out = list(self._y), []
        for _ in range(h):
            x   = np.array(buf[-self.lookback:], float).reshape(1, -1)
            p   = float(self._model.predict(x)[0])
            out.append(p)
            buf.append(p)
        return np.array(out)


if HAS_LGB:
    cv_lgb, mape_lgb = run_model_cv(
        lambda y: LightGBMLagModel(lookback=12).fit(y),
        name='LightGBM (lag=12)'
    )
else:
    print('[SKIP] LightGBM not installed.')
    mape_lgb = None

---
## 8. Linear Decomposition Time-Series Models

### 2.10.1 NLinear (Normalisation-Linear)

In [ ]:
class NLinearModel:
    """
    NLinear: subtract last value (normalise), fit OLS, add last value back.
    Removes distribution shift in the lookback window.
    """

    def __init__(self, lookback: int = 4):
        self.lookback = lookback

    def fit(self, y: np.ndarray):
        y  = np.array(y, float)
        n  = len(y)
        lb = self.lookback
        X  = np.c_[[y[i:i+lb] - y[i+lb-1] for i in range(n - lb)],
                   np.ones(n - lb)]
        Y  = np.array([y[i+lb] - y[i+lb-1] for i in range(n - lb)])
        self.W, *_ = lstsq(X, Y, rcond=None)
        self._y    = y.copy()
        return self

    def predict_ar(self, h: int) -> np.ndarray:
        buf, out = list(self._y), []
        lb       = self.lookback
        for _ in range(h):
            w = np.array(buf[-lb:], float)
            p = float((w - w[-1]) @ self.W[:-1] + self.W[-1] + w[-1])
            out.append(p)
            buf.append(p)
        return np.array(out)


# ── Grid search over lookback ─────────────────────────────────────────────────
print('NLinear grid search over lookback ...')
nl_grid_results = {}
for lb in [4, 6, 8, 12, 16]:
    _, m = run_model_cv(lambda y, lb=lb: NLinearModel(lb).fit(y),
                         name=f'NLinear(lb={lb})')
    nl_grid_results[lb] = m

best_nl_lb = min(nl_grid_results, key=nl_grid_results.get)
print(f'\nBest NLinear lookback = {best_nl_lb}  (MAPE {nl_grid_results[best_nl_lb]:.2f}%)')

### 2.10.2 DLinear (Decomposition-Linear)

In [ ]:
def moving_average(series: np.ndarray, kernel: int) -> np.ndarray:
    """Centred moving average with half-window = kernel // 2."""
    h, n = kernel // 2, len(series)
    out  = np.empty(n)
    for i in range(n):
        out[i] = series[max(0, i - h) : min(n, i + h + 1)].mean()
    return out


class DLinearModel:
    """
    DLinear: decompose into trend + residual using moving average,
    then fit separate OLS models for each component.
    """

    def __init__(self, lookback: int = 12, kernel: int = 7):
        self.lookback = lookback
        self.kernel   = kernel

    def fit(self, y: np.ndarray):
        y  = np.array(y, float)
        n  = len(y)
        lb = self.lookback
        tr = moving_average(y, self.kernel)
        re = y - tr

        Xt = np.c_[[tr[i:i+lb] for i in range(n - lb)], np.ones(n - lb)]
        Xr = np.c_[[re[i:i+lb] for i in range(n - lb)], np.ones(n - lb)]

        self.Wt, *_ = lstsq(Xt, [tr[i+lb] for i in range(n - lb)], rcond=None)
        self.Wr, *_ = lstsq(Xr, [re[i+lb] for i in range(n - lb)], rcond=None)
        self._y     = y.copy()
        return self

    def predict_ar(self, h: int) -> np.ndarray:
        buf, out = list(self._y), []
        lb       = self.lookback
        for _ in range(h):
            b  = np.array(buf, float)
            t2 = moving_average(b, self.kernel)
            r2 = b - t2
            p  = float(t2[-lb:] @ self.Wt[:-1] + self.Wt[-1]
                       + r2[-lb:] @ self.Wr[:-1] + self.Wr[-1])
            out.append(p)
            buf.append(p)
        return np.array(out)


class MultiDLinearModel:
    """Ensemble of DLinear models with multiple lookbacks — averages predictions."""

    def __init__(self, lookbacks=(4, 8, 12, 16), kernel: int = 5):
        self.lookbacks = lookbacks
        self.kernel    = kernel

    def fit(self, y: np.ndarray):
        self._y = np.array(y, float).copy()
        self._models = [
            DLinearModel(lb, self.kernel).fit(self._y)
            for lb in self.lookbacks if lb < len(y) - 1
        ]
        return self

    def predict_ar(self, h: int) -> np.ndarray:
        preds = [m.predict_ar(h) for m in self._models]
        return np.mean(preds, axis=0)


# ── Grid search ───────────────────────────────────────────────────────────────
print('DLinear grid search (lookback × kernel) ...')
dl_grid_results = {}
for lb in [6, 8, 10, 12, 16]:
    for k in [5, 7, 9]:
        _, m = run_model_cv(
            lambda y, lb=lb, k=k: DLinearModel(lb, k).fit(y),
            name=f'DLinear(lb={lb},k={k})'
        )
        dl_grid_results[(lb, k)] = m

best_dl = min(dl_grid_results, key=dl_grid_results.get)
print(f'\nBest DLinear: lookback={best_dl[0]}, kernel={best_dl[1]}  '
      f'(MAPE {dl_grid_results[best_dl]:.2f}%)')

cv_dl, mape_dl = run_model_cv(
    lambda y, b=best_dl: DLinearModel(b[0], b[1]).fit(y),
    name=f'DLinear (best)'
)
cv_mdl, mape_mdl = run_model_cv(
    lambda y: MultiDLinearModel((4, 8, 12, 16), 5).fit(y),
    name='MultiDLinear'
)

---
## 9. Ensemble Learning & Model Blending

### 2.11 Model Performance Summary

In [ ]:
# ── Collect all CV MAPEs ──────────────────────────────────────────────────────
MODEL_REGISTRY = {
    'Naive':       (lambda y: NaiveModel().fit(y),                          mape_naive),
    'SeasonalNaive': None,   # handled separately (uses monthly data)
    'SES':         (lambda y: SESModel().fit(y),                            mape_ses),
    'ARIMA':       (lambda y: ARIMAModel((1,1,0)).fit(y),                   mape_arima),
    'AutoSARIMA':  (lambda y: SARIMAModel(auto=HAS_PMDARIMA).fit(y),        mape_sarima),
    'Theta':       (lambda y: ThetaModel().fit(y),                          mape_theta),
    'NLinear':     (lambda y, lb=best_nl_lb: NLinearModel(lb).fit(y),       nl_grid_results[best_nl_lb]),
    'DLinear':     (lambda y, b=best_dl: DLinearModel(b[0],b[1]).fit(y),    mape_dl),
    'MultiDLinear':(lambda y: MultiDLinearModel((4,8,12,16),5).fit(y),      mape_mdl),
}
if HAS_LGB and mape_lgb is not None:
    MODEL_REGISTRY['LightGBM'] = (lambda y: LightGBMLagModel(12).fit(y), mape_lgb)

# ── Summary table ─────────────────────────────────────────────────────────────
summary_rows = [
    {'Model': name, 'CV MAPE (%)': round(val[1], 2)}
    for name, val in MODEL_REGISTRY.items()
    if val is not None
]
summary_df = pd.DataFrame(summary_rows).sort_values('CV MAPE (%)').reset_index(drop=True)
print('\n── Model CV MAPE Summary ──')
print(summary_df.to_string(index=False))

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
colors  = ['#2ecc71' if v == summary_df['CV MAPE (%)'].min() else '#3498db'
           for v in summary_df['CV MAPE (%)']]
ax.barh(summary_df['Model'], summary_df['CV MAPE (%)'], color=colors)
ax.set_xlabel('CV MAPE (%)')
ax.set_title('Walk-Forward CV MAPE by Model')
ax.axvline(5, color='red', linestyle='--', linewidth=1.5, label='5% target')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_model_comparison.png', bbox_inches='tight')
plt.show()

### Weighted Ensemble: 80% Weekly Models + 20% Seasonal Naïve

In [ ]:
# ── Ensemble candidates (exclude Naive, SeasonalNaive, ARIMA — weakest) ───────
ENSEMBLE_MODELS = {
    k: v for k, v in MODEL_REGISTRY.items()
    if v is not None and k not in ('Naive', 'SeasonalNaive', 'ARIMA')
}

# ── 1/MAPE weights ────────────────────────────────────────────────────────────
raw_inv = {name: 1.0 / max(v[1], 0.01) for name, v in ENSEMBLE_MODELS.items()}
total   = sum(raw_inv.values())
WEIGHTS = {name: raw_inv[name] / total for name in ENSEMBLE_MODELS}

print('Ensemble weights (1/MAPE, normalised):')
for name, w in sorted(WEIGHTS.items(), key=lambda x: -x[1]):
    print(f'  {name:<16}: {w:.4f}  (CV {ENSEMBLE_MODELS[name][1]:.2f}%)')

ALPHA_SN = 0.20   # seasonal naive blend fraction
print(f'\nBlend: {(1-ALPHA_SN)*100:.0f}% ensemble + {ALPHA_SN*100:.0f}% seasonal naïve')

In [ ]:
def ensemble_forecast_month(ym: tuple,
                             mask: np.ndarray,
                             fws: list, fwe: list) -> tuple:
    """
    Run the full ensemble + seasonal-naïve blend for a single target month.
    Returns (pred_freq, pred_total).
    """
    tr_f = prepare_series(FREQ_RAW[mask], use_log=False)
    tr_t = prepare_series(TOT_RAW[mask],  use_log=True)
    h    = len(fws)

    ens_f = np.zeros(h)
    ens_t = np.zeros(h)

    for name, (fn, _) in ENSEMBLE_MODELS.items():
        pf = np.maximum(fn(tr_f).predict_ar(h), 0)
        pt = exp_transform(fn(tr_t).predict_ar(h))
        ens_f += WEIGHTS[name] * pf
        ens_t += WEIGHTS[name] * pt

    rof = weekly_to_monthly_rollup(fws, fwe, ens_f, {ym})
    rot = weekly_to_monthly_rollup(fws, fwe, ens_t, {ym})
    pf_ens, pt_ens = rof[ym], rot[ym]

    # Seasonal naïve component (uses monthly history up to mask)
    idx    = list(monthly['ym']).index(ym) if ym in list(monthly['ym']) else len(monthly)
    tr_mg  = mg_agg.iloc[:idx]
    pf_sn, pt_sn = seasonal_naive_predict(tr_mg, ym)
    if pf_sn is None:
        pf_sn, pt_sn = pf_ens, pt_ens

    pf = (1 - ALPHA_SN) * pf_ens + ALPHA_SN * pf_sn
    pt = (1 - ALPHA_SN) * pt_ens + ALPHA_SN * pt_sn
    return pf, pt


# ── Backtest the full blend ───────────────────────────────────────────────────
print('Backtest — Weighted Ensemble + Seasonal Naïve blend')
print('=' * 72)
print(f"{'Month':<14}{'GT_Freq':>8}{'Pred_Freq':>10}{'APE_F':>7} | "
      f"{'GT_Sev':>12}{'Pred_Sev':>13}{'APE_S':>7} | {'APE_T':>7}")
print('-' * 84)

blend_results = []
all_e = []

for ym in HELD_OUT:
    ms   = pd.Timestamp(year=ym[0], month=ym[1], day=1)
    me   = ms + pd.offsets.MonthEnd(0)
    mask = WEEKLY_WE_PD < ms
    fws, fwe = get_future_weeks(ms, me)

    pf, pt = ensemble_forecast_month(ym, mask, fws, fwe)
    ps     = pt / pf if pf > 0 else 0
    gt     = MONTHLY_GT[ym]

    ef = abs(gt['frequency']   - pf) / gt['frequency']   * 100
    es = abs(gt['severity']    - ps) / gt['severity']    * 100
    et = abs(gt['total_claim'] - pt) / gt['total_claim'] * 100

    all_e += [ef, es, et]
    blend_results.append({'ym': ym, 'freq_ape': ef, 'sev_ape': es, 'tot_ape': et})

    print(f"{str(ym):<14}{gt['frequency']:>8.0f}{pf:>10.1f}{ef:>6.2f}% | "
          f"{gt['severity']:>12.0f}{ps:>13.0f}{es:>6.2f}% | {et:>6.2f}%")

print('-' * 84)
mape_blend = np.mean(all_e)
print(f"{'OVERALL MAPE':>62}: {mape_blend:.4f}%")

---
## 10. Final Predictions & Submission

In [ ]:
# ── Future weeks: Aug 4 → Dec 31, 2025 ───────────────────────────────────────
last_week_end = weekly['week_end'].max()
cursor        = last_week_end + pd.Timedelta(days=1)
FWS, FWE      = [], []
while cursor <= pd.Timestamp('2025-12-31'):
    FWS.append(cursor)
    FWE.append(min(cursor + pd.Timedelta(days=6), pd.Timestamp('2025-12-31')))
    cursor += pd.Timedelta(days=7)

TARGET_YMS = {(2025, m) for m in range(8, 13)}
H          = len(FWS)
print(f'Future weeks to predict: {H}')

# ── Full-data ensemble ────────────────────────────────────────────────────────
tr_f_full = prepare_series(FREQ_RAW, use_log=False)
tr_t_full = prepare_series(TOT_RAW,  use_log=True)

ens_f_full = np.zeros(H)
ens_t_full = np.zeros(H)

for name, (fn, _) in ENSEMBLE_MODELS.items():
    pf = np.maximum(fn(tr_f_full).predict_ar(H), 0)
    pt = exp_transform(fn(tr_t_full).predict_ar(H))
    ens_f_full += WEIGHTS[name] * pf
    ens_t_full += WEIGHTS[name] * pt
    print(f'  {name}: done')

mo_f_ens = weekly_to_monthly_rollup(FWS, FWE, ens_f_full, TARGET_YMS)
mo_t_ens = weekly_to_monthly_rollup(FWS, FWE, ens_t_full, TARGET_YMS)

# ── Add actual Aug 1-3 (last training week overlaps Aug 1-3) ─────────────────
last_w = weekly.iloc[-1]
mo_f_ens[(2025, 8)] += float(last_w['frequency'])  * 3 / 7
mo_t_ens[(2025, 8)] += float(last_w['total_claim']) * 3 / 7

print('Ensemble predictions done.')

In [ ]:
# ── Seasonal naive on full monthly history ────────────────────────────────────
LABEL_MAP = {
    (2025, 8):  '2025_08',
    (2025, 9):  '2025_09',
    (2025, 10): '2025_10',
    (2025, 11): '2025_11',
    (2025, 12): '2025_12',
}

print('\n── Final Predictions (80% ensemble + 20% seasonal naïve) ──')
print(f"{'Month':<10}{'Frequency':>12}{'Severity (IDR)':>20}{'Total Claim (IDR)':>22}")
print('-' * 66)

submission_rows = []

for ym, lbl in sorted(LABEL_MAP.items()):
    pf_ens = mo_f_ens[ym]
    pt_ens = mo_t_ens[ym]

    pf_sn, pt_sn = seasonal_naive_predict(mg_agg, ym)
    if pf_sn is None:
        pf_sn, pt_sn = pf_ens, pt_ens

    f = (1 - ALPHA_SN) * pf_ens + ALPHA_SN * pf_sn
    t = (1 - ALPHA_SN) * pt_ens + ALPHA_SN * pt_sn
    s = t / f if f > 0 else 0

    print(f"{lbl:<10}{f:>12.1f}{s:>20,.0f}{t:>22,.0f}")

    submission_rows += [
        {'id': f'{lbl}_Claim_Frequency', 'value': round(f, 2)},
        {'id': f'{lbl}_Claim_Severity',  'value': round(s, 2)},
        {'id': f'{lbl}_Total_Claim',     'value': round(t, 2)},
    ]

submission = pd.DataFrame(submission_rows)
submission.to_csv(OUTPUT_DIR / 'predictions_2025.csv', index=False)
print(f'\nSaved → {OUTPUT_DIR / "predictions_2025.csv"}')
print(f'CV MAPE (blend backtest): {mape_blend:.4f}%')

In [ ]:
# ── Forecast visualisation ────────────────────────────────────────────────────
months_hist = [pd.Timestamp(year=y, month=m, day=1) for y, m in monthly['ym']]
months_pred = [pd.Timestamp(year=y, month=m, day=1) for y, m in sorted(LABEL_MAP.keys())]

pred_freq  = [submission.loc[submission['id'] == f'{lbl}_Claim_Frequency', 'value'].values[0]
              for lbl in [LABEL_MAP[k] for k in sorted(LABEL_MAP)]]
pred_tot   = [submission.loc[submission['id'] == f'{lbl}_Total_Claim',     'value'].values[0]
              for lbl in [LABEL_MAP[k] for k in sorted(LABEL_MAP)]]

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# Frequency
axes[0].plot(months_hist, monthly['frequency'], marker='o', color='steelblue', label='Historical')
axes[0].plot(months_pred, pred_freq, marker='s', linestyle='--', color='tomato', label='Forecast')
axes[0].set_title('Monthly Claim Frequency — Historical vs Forecast')
axes[0].set_ylabel('# Claims')
axes[0].legend()

# Total Claim
axes[1].plot(months_hist, monthly['total_claim'] / 1e9, marker='o', color='steelblue', label='Historical')
axes[1].plot(months_pred, [v / 1e9 for v in pred_tot], marker='s', linestyle='--', color='tomato', label='Forecast')
axes[1].set_title('Monthly Total Claim — Historical vs Forecast (B IDR)')
axes[1].set_ylabel('Total Claim (B IDR)')
axes[1].legend()

for ax in axes:
    ax.axvline(pd.Timestamp('2025-08-01'), color='gray', linestyle=':', linewidth=1.5, label='Forecast start')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_final_forecast.png', bbox_inches='tight')
plt.show()

print('\n── Final submission preview ──')
print(submission.to_string(index=False))